# Resample data


---

### Import libraries

In [ ]:
import sys
import os


root = os.path.abspath('../..')  
sys.path.append(root)

import pandas as pd
import numpy as np
from scipy.interpolate import PchipInterpolator
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')

from modules import processing, load, plots, analysis

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 25)

---

### Import data

In [ ]:
well_name = 'BW5D_YSI_20230822'

df_well = pd.read_csv(f'{root}/data/rawdy/{well_name}_rowdy.csv')

# Renombrar las columnas para que coincidan con los nombres esperados
if 'Vertical Position [m]' in df_well.columns and 'Corrected sp Cond [uS/cm]' in df_well.columns:
    df_well = df_well.rename(columns={
        'Vertical Position [m]': 'Vertical Position m',
        'Corrected sp Cond [uS/cm]': 'Corrected sp Cond [µS/cm]'
    })

df_well = df_well[['Vertical Position m', 'Corrected sp Cond [µS/cm]']]


df_well

---

### Analyze spacing between points

In [ ]:
def analize_dz(depths, 
               percentile=95):
    """
    Estimate the optimal sampling interval (dz) from a 1D depth array and produce a boxplot.

    Parameters
    ----------
    depths : array-like (1D)
        Depth measurements (in meters). Can be a pandas Series/DataFrame or numpy array.
    percentile : int, optional
        Percentile to use when estimating dz. Default is 95.
    well_name : str, optional
        Name of the well (for plot title). Default is "Well".

    Returns
    -------
    stats_df : pd.DataFrame
        Summary statistics for dz (percentile, median, mean, min, max, dz_max).
    fig : plotly.graph_objects.Figure
        Boxplot figure showing the distribution of Δz with metric lines.
    """
    # Coerce to 1D numpy array and drop NaNs
    if isinstance(depths, pd.DataFrame):
        depths = depths.iloc[:, 0]
    arr = np.asarray(depths).ravel()
    arr = arr[~np.isnan(arr)]

    sorted_depths = np.sort(arr)
    delta_z = np.diff(sorted_depths)

    # Compute summary statistics
    pval = np.percentile(delta_z, percentile)
    median = np.median(delta_z)
    mean = np.mean(delta_z)
    mn = np.min(delta_z)
    mx = np.max(delta_z)

    stats_df = pd.DataFrame({
        f'percentile{percentile}': [pval],
        'median': [median],
        'mean': [mean],
        'min': [mn],
        'max': [mx]
    })

    # Build boxplot
    fig = go.Figure()
    fig.add_trace(go.Box(
        y=delta_z,
        name='Δz',
        boxpoints='outliers',
        marker_color='rgb(8,81,156)',
        line_color='rgb(8,81,156)'
    ))

    # Overlay horizontal lines for each metric
    metrics = {
        f'Percentile {percentile}': pval,
        'Median': median,
        'Mean': mean,
        'Min': mn,
        'Max': mx
    }
    colors = {
        f'Percentile {percentile}': 'red',
        'Median': 'green',
        'Mean': 'yellow',
        'Min': 'purple',
        'Max': 'orange'
    }
    for label, value in metrics.items():
        fig.add_hline(
            y=value,
            line_dash="dash",
            line_color=colors[label],
            annotation_text=f"{label}: {value:.3f} m",
            annotation_position="right"
        )

    # Layout adjustments
    fig.update_layout(
        title=f'Distribution of sampling intervals (Δz): {well_name}',
        yaxis_title='Δz (meters)',
        showlegend=False,
        template='plotly_white',
        height=800,
        width=700,
        margin=dict(r=150)
    )

    return stats_df, fig

In [ ]:
stats_dz, fig = analize_dz(depths = df_well['Vertical Position m'], percentile=95)

fig.show()

stats_dz

---

### Resample data

In [ ]:
def resample_profile(
    df: pd.DataFrame,
    depth_col: str = 'Vertical Position m',
    ec_col: str = 'Corrected sp Cond [µS/cm]', 
    dz: float = None,
    dz_method: str = 'percentile95',
    adaptive_refinement: bool = False,
    sort_values: bool = False
) -> pd.DataFrame:
    """
    Resample a depth vs conductivity profile:
      1) Clean and optionally sort
      2) Create uniform depth grid
      3) Monotonic PCHIP interpolation
      4) Optional adaptive refinement in high-gradient zones

    Parameters
    ----------
    df : pandas.DataFrame
        Original data with depth and conductivity columns.
    depth_col : str
        Name of the depth column (meters).
    ec_col : str
        Name of the conductivity column (µS/cm).
    dz : float, optional
        Desired grid spacing. If None, calculated automatically.
    dz_method : str
        Method to estimate dz if not provided: 'percentile95', 'median', 'mean', 'min'.
    adaptive_refinement : bool
        If True, add midpoints in regions with steep gradients.
    sort_values : bool
        If True, sort data by depth before processing.

    Returns
    -------
    pandas.DataFrame or tuple
        If adaptive_refinement is False: DataFrame with columns:
          depth_col and ec_col sampled on uniform grid.
        If adaptive_refinement is True: tuple of:
          - full resampled DataFrame
          - DataFrame of added points (depth and conductivity)
    """
    # 1. Clean and sort
    data = df[[depth_col, ec_col]].dropna()
    if sort_values:
        data = data.sort_values(depth_col)

    depths = data[depth_col].values
    ecs = data[ec_col].values

    # 2. Determine dz_target
    if dz is None:
        dz_target = analize_dz(depths, percentile=95)[0]
        dz_target = float(dz_target[dz_method])
    else:
        dz_target = dz

    # 3. Build uniform depth grid including endpoints
    z_min, z_max = depths[0], depths[-1]
    uniform_grid = np.arange(z_min, z_max, dz_target)
    # Ensure z_max included
    if uniform_grid[-1] < z_max - 1e-10:
        uniform_grid = np.append(uniform_grid, z_max)

    # 4. PCHIP interpolation
    interpolator = PchipInterpolator(depths, ecs)
    ec_uniform = interpolator(uniform_grid)

    # 5. Adaptive refinement
    added_points = None
    if adaptive_refinement:
        # Compute absolute gradient
        grad = np.abs(np.gradient(ec_uniform, uniform_grid))
        threshold = 3 * np.median(grad)
        # Find indices where gradient exceeds threshold (excluding last index)
        high_grad_idx = np.where(grad > threshold)[0]
        # Only consider those with a next neighbor
        valid = high_grad_idx[high_grad_idx < len(uniform_grid) - 1]
        if valid.size > 0:
            # Compute midpoints
            mids = (uniform_grid[valid] + uniform_grid[valid + 1]) / 2.0
            # Unique and sorted
            mids = np.unique(mids)
            # Evaluate at mids
            ec_mids = interpolator(mids)
            # Merge grids
            final_grid = np.sort(np.concatenate([uniform_grid, mids]))
            ec_final = interpolator(final_grid)
            added_points = pd.DataFrame({depth_col: mids, ec_col: ec_mids})
        else:
            final_grid, ec_final = uniform_grid, ec_uniform
    else:
        final_grid, ec_final = uniform_grid, ec_uniform

    # 6. Package full result
    full_df = pd.DataFrame({depth_col: final_grid, ec_col: ec_final})

    if adaptive_refinement:
        return full_df, added_points
    
    return full_df

def create_comparison_plot(original_df: pd.DataFrame, resampled_df: pd.DataFrame, 
                          sampling_interval: float, x_min: float, num_intervals: int,
                          adaptive_points: pd.DataFrame = None) -> go.Figure:
    """
    Create a comparison plot between original and resampled conductivity data.
    
    Args:
        original_df: DataFrame with original conductivity data
        resampled_df: DataFrame with resampled conductivity data
        sampling_interval: Interval used for resampling (in meters)
        x_min: Minimum x-value for inset plot
        num_intervals: Number of intervals to show in inset plot
        adaptive_points: Optional DataFrame with adaptively refined points
        
    Returns:
        Plotly figure object with main plot and inset
    """
    # Create main plot with subplots
    fig = make_subplots(specs=[[{"secondary_y": False}]])
    
    # Add resampled data points
    fig.add_trace(
        go.Scatter(
            x=resampled_df['Vertical Position m'],
            y=resampled_df['Corrected sp Cond [µS/cm]'],
            mode='markers',
            marker=dict(size=10, color='red'),
            name=f'Resampled ({sampling_interval*100:.0f} cm) n = {len(resampled_df)}'
        )
    )

    # Add original data points
    fig.add_trace(
        go.Scatter(
            x=original_df['Vertical Position m'],
            y=original_df['Corrected sp Cond [µS/cm]'],
            mode='markers',
            marker=dict(size=4, color='blue'),
            name=f'Original data n = {len(original_df)}'
        )
    )
    
    # Add adaptive refinement points if provided
    if adaptive_points is not None:
        fig.add_trace(
            go.Scatter(
                x=adaptive_points['Vertical Position m'],
                y=adaptive_points['Corrected sp Cond [µS/cm]'],
                mode='markers',
                marker=dict(size=8, color='green'),
                name=f'Adaptive points n = {len(adaptive_points)}'
            )
        )
    
    # Configure main layout
    fig.update_layout(
        title=f'Comparison of original vs resampled data {well_name}',
        xaxis_title='Vertical Position (m)',
        yaxis_title='Corrected Specific Conductivity (µS/cm)',
        legend=dict(x=0.75, y=0.1),
        width=1000,
        height=600,
        hovermode='closest'
    )
    
    # Define inset plot parameters
    x_max = x_min + num_intervals * sampling_interval
    mask = (original_df['Vertical Position m'] >= x_min) & (original_df['Vertical Position m'] <= x_max)
    y_min = original_df.loc[mask, 'Corrected sp Cond [µS/cm]'].min()
    y_max = original_df.loc[mask, 'Corrected sp Cond [µS/cm]'].max()
    margin = (y_max - y_min) * 0.1
    
    # Add rectangle for inset
    fig.add_shape(
        type="rect",
        x0=0.05, y0=0.65,
        x1=0.35, y1=0.95,
        xref="paper", yref="paper",
        line=dict(color="black", width=1),
        fillcolor="white",
        opacity=0.7
    )
    
    # Add original data to inset
    fig.add_trace(
        go.Scatter(
            x=original_df.loc[mask, 'Vertical Position m'],
            y=original_df.loc[mask, 'Corrected sp Cond [µS/cm]'],
            mode='markers',
            marker=dict(size=4, color='blue'),
            showlegend=False,
            xaxis="x2", yaxis="y2"
        )
    )
    
    # Add resampled data to inset
    inset_mask = (resampled_df['Vertical Position m'] >= x_min) & (resampled_df['Vertical Position m'] <= x_max)
    fig.add_trace(
        go.Scatter(
            x=resampled_df.loc[inset_mask, 'Vertical Position m'],
            y=resampled_df.loc[inset_mask, 'Corrected sp Cond [µS/cm]'],
            mode='markers',
            marker=dict(size=10, color='red'),
            showlegend=False,
            xaxis="x2", yaxis="y2"
        )
    )

    # Add adaptive points to inset if provided
    if adaptive_points is not None:
        inset_mask_adaptive = (adaptive_points['Vertical Position m'] >= x_min) & (adaptive_points['Vertical Position m'] <= x_max)
        fig.add_trace(
            go.Scatter(
                x=adaptive_points.loc[inset_mask_adaptive, 'Vertical Position m'],
                y=adaptive_points.loc[inset_mask_adaptive, 'Corrected sp Cond [µS/cm]'],
                mode='markers',
                marker=dict(size=8, color='green'),
                showlegend=False,
                xaxis="x2", yaxis="y2"
            )
        )
    
    # Add vertical dotted lines in inset to show sampling intervals
    for i in range(1, num_intervals):
        x_line = x_min + i*sampling_interval
        fig.add_shape(
            type="line",
            x0=x_line, y0=y_min - margin,
            x1=x_line, y1=y_max + margin,
            xref="x2", yref="y2",
            line=dict(color="gray", width=1, dash="dot")
        )
    
    # Configure inset axes
    fig.update_layout(
        xaxis2=dict(
            domain=[0.05, 0.35],
            anchor="y2",
            range=[x_min, x_max]
        ),
        yaxis2=dict(
            domain=[0.65, 0.95],
            anchor="x2",
            range=[y_min - margin, y_max + margin],
            showticklabels=False
        )
    )
    
    return fig

def main(df: pd.DataFrame,
         dz: float = 0.05,
         x_min: float = 0.0,
         num_intervals: int = 10,
         adaptive_refinement: bool = False,
         sort_values: bool = False,
         depth_col: str = 'Vertical Position m',
         ec_col: str = 'Corrected sp Cond [µS/cm]',
         dz_method: str = 'percentile95') -> tuple:
    """
    Main function that integrates resampling and plotting functionality.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Original data with depth and conductivity columns
    dz : float
        Desired grid spacing in meters
    x_min : float
        Minimum x-value for inset plot
    num_intervals : int
        Number of intervals to show in inset plot
    adaptive_refinement : bool
        If True, use adaptive refinement
    sort_values : bool
        If True, sort data by depth
    depth_col : str
        Name of depth column
    ec_col : str
        Name of conductivity column
    dz_method : str
        Method to estimate dz if not provided
        
    Returns
    -------
    tuple
        (resampled_df, figure) or (resampled_df, added_points, figure)
    """
    # Perform resampling
    resampled_result = resample_profile(
        df=df,
        depth_col=depth_col,
        ec_col=ec_col,
        dz=dz,
        dz_method=dz_method,
        adaptive_refinement=adaptive_refinement,
        sort_values=sort_values
    )
    
    # Handle adaptive refinement case
    if adaptive_refinement:
        resampled_df, added_points = resampled_result
        fig = create_comparison_plot(
            original_df=df,
            resampled_df=resampled_df,
            sampling_interval=dz,
            x_min=x_min,
            num_intervals=num_intervals,
            adaptive_points=added_points
        )
        return resampled_df, added_points, fig
    else:
        resampled_df = resampled_result
        fig = create_comparison_plot(
            original_df=df,
            resampled_df=resampled_df,
            sampling_interval=dz,
            x_min=x_min,
            num_intervals=num_intervals
        )
        return resampled_df, fig


---

### 1. Uniform resample data

In [ ]:
resample , fig = main(df = df_well,
                            dz = 0.05, #meters
                            x_min = 10,
                            num_intervals = 10,
                            adaptive_refinement = False,
                            sort_values = False
                            # dz_method='percentile95'
                        )

fig.show()

---

### 2. ADAPTIVE REFINEMENT (NON-UNIFORM RESAMPLING)

In [ ]:
resample, refin, fig = main(df = df_well,
                            dz = 0.05, #meters
                            x_min = 10,
                            num_intervals = 10,
                            adaptive_refinement = True,
                            sort_values = False
                        )

fig.show()

#### Data adapatative refinement

In [ ]:
refin